# Spatial-ring symWeighted — exact dual factorisation gate

This fail-closed notebook runs one immutable judge in normal and optimised Python. It licenses only one exact entrywise factorisation of the ring-weighted kernel; it does not license Jordan–Wigner, a norm estimate, or either sector bound.

In [ ]:
import datetime, hashlib, json, os, platform, subprocess, sys, tempfile, urllib.request
from pathlib import Path

RAW_COMMIT = '06226edc9221fa60a6ed39e30ae84c848bd66041'
RAW_URL = ('https://raw.githubusercontent.com/lluiseriksson/'
    f'THE-ERIKSSON-PROGRAMME/{RAW_COMMIT}/scripts/judge_spatial_symweighted_factorization.py')
EXPECTED_SHA256 = 'a95e66da0ee527b1776ceb3d13d83760d1fd88cc9227ebea668a2b98ca1946cf'
EXPECTED_PAIRS = sum(4**size for size in range(1, 7))

run_root = Path(tempfile.mkdtemp(prefix='spatial-symweighted-factorization-'))
judge = run_root / 'judge_spatial_symweighted_factorization.py'
source = urllib.request.urlopen(RAW_URL, timeout=60).read()
source_hash = hashlib.sha256(source).hexdigest()
if source_hash != EXPECTED_SHA256:
    raise RuntimeError(f'judge SHA-256 mismatch: {source_hash}')
judge.write_bytes(source)

record = {
  'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'python': platform.python_version(),
  'cpu_count': os.cpu_count(), 'raw_commit': RAW_COMMIT,
  'judge_sha256': source_hash, 'runs': [],
}
required_counters = (
    'configuration_pairs_checked', 'scale_mutations_rejected',
    'source_closing_bond_mutations_rejected',
    'target_closing_bond_mutations_rejected',
)
for flags in ([], ['-O']):
    command = [sys.executable, *flags, str(judge)]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, check=False)
    print('$', ' '.join(command))
    print(result.stdout, end='')
    print(f'[exit {result.returncode}]')
    if result.returncode != 0:
        raise RuntimeError(f'judge failed under flags {flags}')
    payload = json.loads(result.stdout)
    if payload.get('status') != 'PASS':
        raise RuntimeError(f'judge omitted PASS under flags {flags}')
    for counter in required_counters:
        if payload.get(counter) != EXPECTED_PAIRS:
            raise RuntimeError(f'wrong {counter} under flags {flags}: {payload}')
    record['runs'].append({'flags': flags, 'exit': result.returncode, 'payload': payload})

artifact = run_root / 'spatial_symweighted_factorization_gate.json'
artifact.write_text(json.dumps(record, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(f'artifact_sha256={hashlib.sha256(artifact.read_bytes()).hexdigest()}')
print('SPATIAL SYMWEIGHTED-FACTORIZATION GATE PASS')
from google.colab import files
files.download(str(artifact))